# Load and inspect

In [15]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

df=pd.read_csv(r"C:\Users\YADVEER\OneDrive\Desktop\churn-analysis\data\WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df.info())
print(df.isna().sum())
print("Duplicates: ",df.duplicated().sum())
print(df["Churn"].value_counts(normalize=True).round(3))


(7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null

# Find the hidden blanks

In [16]:
blank =df[df['TotalCharges'].str.strip() == ""]
print(len(blank))
blank[["customerID","tenure","TotalCharges","Churn"]]

11


,customerID,tenure,TotalCharges,Churn
488,4472-LVYGI,0,,No
753,3115-CZMZD,0,,No
936,5709-LVOEQ,0,,No
1082,4367-NUYAO,0,,No
1340,1371-DWPAZ,0,,No
3331,7644-OMVMY,0,,No
3826,3213-VVOLG,0,,No
4380,2520-SGTTA,0,,No
5218,2923-ARZLG,0,,No
6670,4075-WKNIU,0,,No


# Clean

In [17]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
).fillna(0)

df["Churn"] = df["Churn"].astype(str).str.strip().map({"Yes": 1, "No": 0})

svc = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
for c in svc:
    df[c]=df[c].replace("No internet service","No")

df["MultipleLines"]=df['MultipleLines'].replace("No phone service","No")
df["tenure_group"]= pd.cut(
    df['tenure'],
    [-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)
df["num_services"] = (df[svc] == "Yes").sum(axis=1)

df.to_csv("../data/telco_clean.csv", index=False)

df.shape


(7043, 23)

# Quick check

In [18]:
print(df["TotalCharges"].dtype, df["Churn"].mean())
print(df["tenure_group"].value_counts().sort_index())

float64 0.2653698707936959
tenure_group
0-12     2186
13-24    1024
25-48    1594
49-72    2239
Name: count, dtype: int64


# Cleaning decisions
- TotalCharges was text. 11 blanks all had tenure 0 (new customers, not yet billed), so I set them to 0.
- Churn converted from Yes/No to 1/0 for calculations.
- "No internet service" and "No phone service" merged into "No", since they mean the same for analysis.
- Created tenure_group and num_services for segmentation.
- No duplicate rows found.